In [ ]:
import os
import tensorflow as tf

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    f.write('{"username":"claudesistemas","key":"a777dbe64b88859696c89ae59a328930"}')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

print("📥 Descargando LC25000 desde Kaggle...")
os.system('kaggle datasets download -d andrewmvd/lung-and-colon-cancer-histopathological-images -p /content/data_pulmon --unzip')
print("✅ Descarga completa")

print(f"✅ TensorFlow: {tf.__version__}")
print(f"✅ GPU: {tf.config.list_physical_devices('GPU')}")

# Ver estructura
for root, dirs, files in os.walk('/content/data_pulmon'):
    level = root.replace('/content/data_pulmon', '').count(os.sep)
    if level < 5:
        indent = ' ' * 2 * level
        print(f"{indent}{os.path.basename(root)}/ ({len(files)} archivos)")

📥 Descargando LC25000 desde Kaggle...
✅ Descarga completa
✅ TensorFlow: 2.20.0
✅ GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
data_pulmon/ (0 archivos)
  lung_colon_image_set/ (0 archivos)
    colon_image_sets/ (0 archivos)
      colon_aca/ (5000 archivos)
      colon_n/ (5000 archivos)
    lung_image_sets/ (0 archivos)
      lung_scc/ (5000 archivos)
      lung_aca/ (5000 archivos)
      lung_n/ (5000 archivos)


In [ ]:
# ============================================================
# ENTRENAMIENTO CÁNCER DE PULMÓN — EfficientNetB0
# Dataset: LC25000 — 15,000 imágenes de pulmón
# Autor: Danner Jamanca
# ============================================================

import tensorflow as tf
import numpy as np
import os
import shutil
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print(f"✅ TensorFlow: {tf.__version__}")
print(f"✅ GPU: {tf.config.list_physical_devices('GPU')}")

# ============================================================
# 1. ORGANIZAR IMÁGENES
# ============================================================
BASE = '/content/data_pulmon/lung_colon_image_set/lung_image_sets'
OUTPUT = '/content/dataset_pulmon_procesado'

if os.path.exists(OUTPUT):
    shutil.rmtree(OUTPUT)

def recolectar_imagenes(path):
    imagenes = []
    for root, dirs, files in os.walk(path):
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                imagenes.append(os.path.join(root, f))
    return imagenes

normales  = recolectar_imagenes(f'{BASE}/lung_n')
anormales = recolectar_imagenes(f'{BASE}/lung_aca') + recolectar_imagenes(f'{BASE}/lung_scc')

print(f"✅ Normales: {len(normales)} | Anormales: {len(anormales)}")

# Split 70/15/15
n_train, n_temp = train_test_split(normales,  test_size=0.30, random_state=42)
n_val,   n_test = train_test_split(n_temp,    test_size=0.50, random_state=42)
a_train, a_temp = train_test_split(anormales, test_size=0.30, random_state=42)
a_val,   a_test = train_test_split(a_temp,    test_size=0.50, random_state=42)

splits = {
    'train':      {'normal': n_train, 'anormal': a_train},
    'validation': {'normal': n_val,   'anormal': a_val},
    'test':       {'normal': n_test,  'anormal': a_test},
}

for split, clases in splits.items():
    for clase, archivos in clases.items():
        dest = f'{OUTPUT}/{split}/{clase}'
        os.makedirs(dest, exist_ok=True)
        for src in archivos:
            shutil.copy2(src, dest)

print("\n✅ Dataset organizado:")
for split in ['train', 'validation', 'test']:
    for clase in ['normal', 'anormal']:
        n = len(os.listdir(f'{OUTPUT}/{split}/{clase}'))
        print(f"   {split}/{clase}: {n}")

# ============================================================
# 2. GENERADORES
# ============================================================
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
SEED       = 42

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.15,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

val_test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_gen = train_datagen.flow_from_directory(
    f'{OUTPUT}/train', target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='binary', seed=SEED
)
val_gen = val_test_datagen.flow_from_directory(
    f'{OUTPUT}/validation', target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='binary', seed=SEED
)
test_gen = val_test_datagen.flow_from_directory(
    f'{OUTPUT}/test', target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='binary', shuffle=False
)

print(f"\n✅ Clases: {train_gen.class_indices}")
print(f"✅ Train: {train_gen.samples} | Val: {val_gen.samples} | Test: {test_gen.samples}")

# ============================================================
# 3. CLASS WEIGHTS
# ============================================================
labels = train_gen.classes
class_weights = compute_class_weight('balanced', classes=np.unique(labels), y=labels)
class_weight_dict = dict(enumerate(class_weights))
print(f"\n✅ Class weights: {class_weight_dict}")

# ============================================================
# 4. MODELO
# ============================================================
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.4)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)
print(f"\n✅ Modelo creado: {model.count_params():,} parámetros")

# ============================================================
# 5. FASE 1 — Solo cabeza
# ============================================================
print("\n🚀 FASE 1 — Entrenando cabeza...")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks_f1 = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
]

model.fit(
    train_gen, validation_data=val_gen,
    epochs=20, callbacks=callbacks_f1,
    class_weight=class_weight_dict, verbose=1
)

# ============================================================
# 6. FASE 2 — Fine tuning
# ============================================================
print("\n🚀 FASE 2 — Fine tuning últimas 30 capas...")

base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks_f2 = [
    ModelCheckpoint('mejor_modelo_pulmon.keras', monitor='val_accuracy', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=4, min_lr=1e-7, verbose=1),
]

model.fit(
    train_gen, validation_data=val_gen,
    epochs=50, callbacks=callbacks_f2,
    class_weight=class_weight_dict, verbose=1
)

# ============================================================
# 7. EVALUACIÓN
# ============================================================
print("\n📊 Evaluando en test set...")
test_gen.reset()
results = model.evaluate(test_gen, verbose=1)
accuracy = results[1] * 100

print(f"""
╔════════════════════════════════════════╗
║  PRECISIÓN FINAL: {accuracy:.2f}%{' ' * (18 - len(f'{accuracy:.2f}'))}║
╚════════════════════════════════════════╝
""")

# ============================================================
# 8. GUARDAR Y DESCARGAR
# ============================================================
model.save('modelo_pulmon.keras')

from google.colab import files
files.download('mejor_modelo_pulmon.keras')
print("📥 Descargando mejor modelo...")

✅ TensorFlow: 2.20.0
✅ GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
✅ Normales: 5000 | Anormales: 10000

✅ Dataset organizado:
   train/normal: 3500
   train/anormal: 7000
   validation/normal: 750
   validation/anormal: 1500
   test/normal: 750
   test/anormal: 1500
Found 10500 images belonging to 2 classes.
Found 2250 images belonging to 2 classes.
Found 2250 images belonging to 2 classes.

✅ Clases: {'anormal': 0, 'normal': 1}
✅ Train: 10500 | Val: 2250 | Test: 2250

✅ Class weights: {0: np.float64(0.75), 1: np.float64(1.5)}
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

✅ Modelo creado: 4,415,652 parámetros

🚀 FASE 1 — Entrenando cabeza...
Epoch 1/20
329/329 ━━━━━━━━━━━━━━━━━━━━ 275s 728ms/step - accuracy: 0.9771 - loss: 0.0651 - val_accuracy: 0.9982 - val_loss: 0.0048 - learning_rate: 0.0010
Epoch 2/20
329/329 ━━━━━━━━━━━━━━━━━━━━ 196s 597ms/step - accuracy: 0.9880 - loss: 0.0386 - val_accuracy: 0.9969 - val_loss: 0.0064 - learning_rate: 0.0010
Epoc

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Descargando mejor modelo...
